In [1]:
!pip install diffusers transformers accelerate peft

In [2]:
import torch
from diffusers import StableDiffusionXLPipeline, TCDScheduler

In [3]:
# if torch.cuda.is_available():
#   device = torch.device("cuda")
#   print("Using GPU:", torch.cuda.get_device_name(0))
# else:
#   device = torch.device("cpu")
#   print("Using CPU")
base_model_id = "stabilityai/stable-diffusion-xl-base-1.0"
tcd_lora_id = "h1t/TCD-SDXL-LoRA"

In [4]:
# Load the diffusers pipeline for text-to-image generation.
pipe = StableDiffusionXLPipeline.from_pretrained(base_model_id, torch_dtype=torch.float16, variant="fp16")#.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
# Set the scheduler to TCDScheduler.
pipe.scheduler = TCDScheduler.from_config(pipe.scheduler.config)

# Load the TCD-LoRA weights for the model.
pipe.load_lora_weights(tcd_lora_id)
pipe.fuse_lora()

The config attributes {'skip_prk_steps': True} were passed to TCDScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.


In [ ]:
prompt = "Beautiful woman, bubblegum pink, lemon yellow, minty blue, futuristic, high-detail, epic composition, watercolor."

# Perform inference with the pipeline.
image = pipe(
    prompt=prompt,
    num_inference_steps=4,
    guidance_scale=0,
    # Eta (referred to as `gamma` in the paper) is used to control the stochasticity in every step.
    # A value of 0.3 often yields good results.
    # We recommend using a higher eta when increasing the number of inference steps.
    eta=0.3,
    generator=torch.Generator().manual_seed(0),
).images[0]


  0%|          | 0/4 [00:00<?, ?it/s]